# Set operations

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-07-set-operations.ipynb)

A set operation combines the **results** of two queries, row by row. `UNION` stacks them and removes duplicates,
`UNION ALL` stacks them and keeps duplicates, `INTERSECT` keeps rows present in both, and `EXCEPT` keeps rows
present in the first but not the second.

Set operations are the answer to questions such as "customers who ordered in January *and* February" and
"customers who ordered in February *but not* January". This chapter explains the rules, the variants, and the
NULL behaviour interviewers ask about.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

## 1. The rules

The two queries must return the same number of columns, with compatible types. The column names come from the
first query. `ORDER BY` and `LIMIT` apply to the whole combined result, placed after the last query.

To compare two periods, first build one list of customer ids per period. Each list is a query on `orders`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW jan_buyers AS
SELECT DISTINCT customer_id FROM orders WHERE month(order_date) = 1
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW feb_buyers AS
SELECT DISTINCT customer_id FROM orders WHERE month(order_date) = 2
""")

spark.sql("SELECT customer_id FROM jan_buyers ORDER BY customer_id").show()
spark.sql("SELECT customer_id FROM feb_buyers ORDER BY customer_id").show()

## 2. UNION and UNION ALL

`UNION` removes duplicate rows from the combined result. Customers 1, 2 and 3 appear in both months, and `UNION`
lists each once. `UNION ALL` keeps every row, so the same customers appear twice. `UNION ALL` is cheaper, because
it does not need the extra step that removes duplicates. Use it when the inputs cannot overlap, or when the
duplicates are wanted.

In [ ]:
spark.sql("""
SELECT customer_id FROM jan_buyers
UNION
SELECT customer_id FROM feb_buyers
ORDER BY customer_id
""").show()

spark.sql("""
SELECT customer_id FROM jan_buyers
UNION ALL
SELECT customer_id FROM feb_buyers
""").groupBy().count().show()

## 3. INTERSECT and EXCEPT

`INTERSECT` returns the rows in both queries. `EXCEPT` returns the rows in the first query that are not in the
second. The order matters for `EXCEPT`: `A EXCEPT B` and `B EXCEPT A` are different questions.

In [ ]:
spark.sql("""
SELECT customer_id FROM jan_buyers
INTERSECT
SELECT customer_id FROM feb_buyers
ORDER BY customer_id
""").show()

spark.sql("""
SELECT customer_id FROM feb_buyers
EXCEPT
SELECT customer_id FROM jan_buyers
ORDER BY customer_id
""").show()

## 4. The ALL variants

`INTERSECT ALL` and `EXCEPT ALL` keep duplicates and work with counts. A value that appears twice on the left and
once on the right appears once in `EXCEPT ALL`. The next cell uses small inline tables so the counts are visible.

In [ ]:
spark.sql("""
SELECT k FROM VALUES (1), (1), (2) AS a(k)
EXCEPT ALL
SELECT k FROM VALUES (1) AS b(k)
""").show()

## 5. NULL in set operations

Set operations treat two NULLs as equal, which is different from `=` in a `WHERE` clause or a join. Two NULL rows
collapse into one under `UNION`. A join on a NULL key matches nothing.

In [ ]:
spark.sql("""
SELECT x FROM VALUES (CAST(NULL AS INT)) AS a(x)
UNION
SELECT x FROM VALUES (CAST(NULL AS INT)) AS b(x)
""").show()

spark.sql("""
SELECT COUNT(*) AS matched_rows
FROM VALUES (CAST(NULL AS INT)) AS a(x)
JOIN VALUES (CAST(NULL AS INT)) AS b(y) ON a.x = b.y
""").show()

## Interview notes

- **"Customers who did A but not B."** Write it with `EXCEPT`, or with a `LEFT ANTI JOIN`. Both express the same
  set. The anti join is often easier to extend with extra columns.
- **UNION vs UNION ALL.** The usual answer is cost: `UNION` must remove duplicates, which means a sort or a hash
  step over the whole result. Use `UNION ALL` by default and add `DISTINCT` only when duplicates are possible.
- **Column count mismatch** is the most common error. Name the columns in the first query. Spark converts the two
  sides to a common type, so `INT` and `BIGINT` combine without trouble. A value that cannot convert, such as the
  string `'a'` next to an integer, fails the whole query in ANSI mode.

## Practice questions

1. Which customers ordered in both January and February? Sort by `customer_id`.
2. Which customers ordered in February but not in January? Sort by `customer_id`.
3. How many distinct customers ordered in January or February?

## Answers

### Question 1

In [ ]:
q = "SELECT customer_id FROM jan_buyers INTERSECT SELECT customer_id FROM feb_buyers ORDER BY customer_id"
spark.sql(q).show()
assert spark.sql(q).collect() == [(1,), (2,), (3,)]

### Question 2

In [ ]:
q = "SELECT customer_id FROM feb_buyers EXCEPT SELECT customer_id FROM jan_buyers ORDER BY customer_id"
spark.sql(q).show()
assert spark.sql(q).collect() == [(4,), (6,)]

### Question 3

In [ ]:
q = "SELECT COUNT(*) AS distinct_buyers FROM (SELECT customer_id FROM jan_buyers UNION SELECT customer_id FROM feb_buyers)"
spark.sql(q).show()
assert spark.sql(q).collect() == [(6,)]